In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages


# Load datasets
biased = pd.read_csv("./Kaggle_Daten/Student_Performance_Behavior_Dataset/Students_Grading_Dataset_Biased.csv").fillna("Missing")
unbiased = pd.read_csv("./Kaggle_Daten/Student_Performance_Behavior_Dataset/Students_Performance_Dataset.csv").fillna("Missing")

# Identify numeric columns
num_cols_biased = biased.select_dtypes(include=["int64","float64"]).columns.tolist()
num_cols_unbiased = unbiased.select_dtypes(include=["int64","float64"]).columns.tolist()

# Discretization
def discretize(df, cols, bins=4):
    df_disc = df.copy()
    for col in cols:
        if df_disc[col].nunique() <= 1:
            df_disc[col + "_disc"] = 0
        else:
            df_disc[col + "_disc"] = pd.qcut(
                df_disc[col].rank(method="first"), q=bins,
                labels=False, duplicates="drop"
            )
    return df_disc

biased_disc = discretize(biased, num_cols_biased)
unbiased_disc = discretize(unbiased, num_cols_unbiased)

# RST Functions
from collections import defaultdict

def indiscernibility(df, attrs):
    groups = defaultdict(list)
    for i, row in df[attrs].iterrows():
        groups[tuple(row)] .append(i)
    return list(groups.values())

def dependency(df, attrs, decision):
    if not attrs:
        return 0.0
    U = len(df)
    pos = 0
    for block in indiscernibility(df, attrs):
        if df.loc[block, decision].nunique() == 1:
            pos += len(block)
    return pos / U

def quick_reduct(df, attrs, decision):
    R = []
    gamma_star = dependency(df, attrs, decision)
    gamma_R = 0.0
    while gamma_R < gamma_star - 1e-12:
        best_attr = None
        best_gamma = gamma_R
        for a in attrs:
            if a in R: continue
            g = dependency(df, R+[a], decision)
            if g > best_gamma + 1e-12:
                best_gamma = g
                best_attr = a
        if best_attr is None:
            break
        R.append(best_attr)
        gamma_R = best_gamma
    return R

# Conditional attributes
decision_attr = "Grade"

def get_cond_attrs(df):
    conds = []
    for col in df.columns:
        if col == decision_attr: continue
        if col.endswith("_disc") or df[col].dtype == "object":
            conds.append(col)
    for bad in ["Student_ID","Email"]:
        if bad in conds:
            conds.remove(bad)
    return conds

cond_attrs_biased = get_cond_attrs(biased_disc)
cond_attrs_unbiased = get_cond_attrs(unbiased_disc)

biased_reduct = quick_reduct(biased_disc, cond_attrs_biased, decision_attr)
unbiased_reduct = quick_reduct(unbiased_disc, cond_attrs_unbiased, decision_attr)

biased_only = sorted(list(set(biased_reduct) - set(unbiased_reduct)))
unbiased_only = sorted(list(set(unbiased_reduct) - set(biased_reduct)))
common = sorted(list(set(biased_reduct) & set(unbiased_reduct)))

# Create Bias Diagram
plt.figure(figsize=(8,5))
plt.bar(["biased_only","common","unbiased_only"], 
        [len(biased_only), len(common), len(unbiased_only)], color=["red","gray","green"])
plt.title("RST Bias Comparison (Attribute Counts)")
plt.xlabel("Attribute Group")
plt.ylabel("Count")
plot_path = "./Kaggle_Daten/Student_Performance_Behavior_Dataset/rst_bias_diagram.png"
plt.savefig(plot_path)
plt.close()

# PDF Report
pdf_path = "./Kaggle_Daten/Student_Performance_Behavior_Dataset/RST_Bias_Report.pdf"
with PdfPages(pdf_path) as pdf:
    fig, ax = plt.subplots(figsize=(8.27, 11.69))
    ax.axis("off")
    txt = f"""
RST Bias Analysis Report

Biased Reduct: {biased_reduct}
Unbiased Reduct: {unbiased_reduct}

Attributes only in biased reduct:
{biased_only}

Attributes only in unbiased reduct:
{unbiased_only}

Attributes in both reducts:
{common}
"""
    ax.text(0.01, 0.99, txt, va="top", wrap=True)
    pdf.savefig(fig)
    plt.close(fig)

    # Insert diagram
    fig = plt.figure(figsize=(8.27, 11.69))
    img = plt.imread(plot_path)
    plt.imshow(img)
    plt.axis("off")
    pdf.savefig(fig)
    plt.close(fig)

(pdf_path, plot_path)


('./Kaggle_Daten/Student_Performance_Behavior_Dataset/RST_Bias_Report.pdf',
 './Kaggle_Daten/Student_Performance_Behavior_Dataset/rst_bias_diagram.png')